## Part 1: Data + Prototypes (Biological Pipeline)


In [1]:
# ============================================================
# CELL 1 — Mount Drive + Install Dependencies
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')

%pip install scanpy
%pip install pandas
%pip install sklearn

You should consider upgrading via the '/Users/mehdi/Desktop/Computational Genomics Project/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/mehdi/Desktop/Computational Genomics Project/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/mehdi/Desktop/Computational Genomics Project/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.preprocessing import normalize
import os

DATA_DIR = "/Users/mehdi/Desktop/Computational Genomics Project/data" # CHANGE THIS TO YOUR DATA DIRECTORY
OUT_DIR = DATA_DIR
os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
# ============================================================
# CELL 3 — Load Spatial Data
# ============================================================
adata_sp = sc.read_10x_h5(
    f"{DATA_DIR}/CytAssist_Fresh_Frozen_Mouse_Brain_filtered_feature_bc_matrix.h5"
)
print("Spatial data loaded:", adata_sp.shape)
print(adata_sp.var_names[:10])

Spatial data loaded: (4298, 19465)
Index(['Xkr4', 'Rp1', 'Sox17', 'Lypla1', 'Tcea1', 'Rgs20', 'Atp6v1h', 'Oprk1',
       'Npbwr1', 'Rb1cc1'],
      dtype='object')


/Users/mehdi/Desktop/Computational Genomics Project/.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/mehdi/Desktop/Computational Genomics Project/.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [7]:
# ============================================================
# CELL 4 — Load scRNA-seq Data
# ============================================================
adata_sc = sc.read_h5ad(
    f"{DATA_DIR}/90cdd3f7-e61e-43ed-97e9-ddc0f7e87827.h5ad"
)
print("scRNA-seq data loaded:", adata_sc.shape)
print(adata_sc.var_names[:10])
print("Obs columns:", adata_sc.obs.columns.tolist())
print("Cell types:\n", adata_sc.obs["cell_type"].value_counts())

scRNA-seq data loaded: (4167869, 1120)
Index(['ENSMUSG00000024798', 'ENSMUSG00000042385', 'ENSMUSG00000036198',
       'ENSMUSG00000028780', 'ENSMUSG00000015843', 'ENSMUSG00000026768',
       'ENSMUSG00000049928', 'ENSMUSG00000041046', 'ENSMUSG00000032373',
       'ENSMUSG00000004633'],
      dtype='object')
Obs columns: ['donor_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'cluster_id_transfer', 'subclass_transfer', 'cluster_confidence_score', 'subclass_confidence_score', 'high_quality_transfer', 'major_brain_region', 'ccf_region_name', 'brain_section_label', 'tissue_type', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid']
Cell types:
 cell_type
glutamatergic neuron              1219017
GABAergic neuron   

In [8]:
# ============================================================
# CELL 5 — Harmonize Gene Names (uppercase)
# ============================================================

adata_sc.var_names = adata_sc.var["gene_name"].astype(str).str.upper()
adata_sp.var_names = adata_sp.var_names.str.upper()

if "gene_name" in adata_sc.var.columns:
    adata_sc.var = adata_sc.var.drop(columns=["gene_name"])

adata_sc.var.index.name = None
adata_sp.var.index.name = None

print("Gene name harmonization done. ✓")

Gene name harmonization done. ✓


In [9]:
# ============================================================
# CELL 6 — Remove Duplicate Genes
# ============================================================
adata_sc = adata_sc[:, ~adata_sc.var_names.duplicated()].copy()
adata_sp = adata_sp[:, ~adata_sp.var_names.duplicated()].copy()

assert adata_sc.var_names.is_unique, "scRNA-seq still has duplicate genes!"
assert adata_sp.var_names.is_unique, "Spatial still has duplicate genes!"
print("All gene names unique. ✓")

All gene names unique. ✓


In [10]:
# ============================================================
# CELL 7 — Intersect to Shared Genes + Subset
# ============================================================
shared_genes = sorted(
    set(adata_sp.var_names).intersection(set(adata_sc.var_names))
)
print(f"Shared genes: {len(shared_genes)}")

adata_sp = adata_sp[:, shared_genes].copy()
adata_sc = adata_sc[:, shared_genes].copy()

print("After subsetting:")
print("  Spatial:", adata_sp.shape)
print("  scRNA-seq:", adata_sc.shape)

# Save master gene list — gene ORDER is locked from this point on
with open(f"{OUT_DIR}/shared_genes.txt", "w") as f:
    for g in shared_genes:
        f.write(g + "\n")
print("Saved shared_genes.txt ✓")

Shared genes: 1074
After subsetting:
  Spatial: (4298, 1074)
  scRNA-seq: (4167869, 1074)
Saved shared_genes.txt ✓


In [11]:
# ============================================================
# CELL 8 — Normalize scRNA-seq (per cell, BEFORE prototypes)
# ============================================================

# Normalize each cell to 10,000 total counts, then log1p
sc.pp.normalize_total(adata_sc, target_sum=1e4)
sc.pp.log1p(adata_sc)
print("scRNA-seq normalization done. ✓")

/Users/mehdi/Desktop/Computational Genomics Project/.venv/lib/python3.9/site-packages/scanpy/preprocessing/_normalization.py:234: UserWarning: Some cells have zero counts
  warn(UserWarning("Some cells have zero counts"))


scRNA-seq normalization done. ✓


In [12]:
# ============================================================
# CELL 9 — Normalize Spatial Data (same scheme as scRNA-seq)
# ============================================================
sc.pp.normalize_total(adata_sp, target_sum=1e4)
sc.pp.log1p(adata_sp)
print("Spatial normalization done. ✓")

adata_sp.write(f"{OUT_DIR}/adata_sp_processed.h5ad")
print("Saved adata_sp_processed.h5ad ✓")

Spatial normalization done. ✓
Saved adata_sp_processed.h5ad ✓


In [14]:
# ============================================================
# CELL 10 — Build Cell-Type Prototypes
# ============================================================

X = adata_sc.X.toarray() if hasattr(adata_sc.X, "toarray") else np.array(adata_sc.X)
print("Expression matrix shape:", X.shape)

df = pd.DataFrame(
    X,
    index=adata_sc.obs["cell_type"].values,
    columns=adata_sc.var_names
)

prototypes_raw = df.groupby(level=0, observed=False).mean()
print(f"Prototype matrix: {prototypes_raw.shape}")
print(f"Cell types: {prototypes_raw.index.tolist()}")


Expression matrix shape: (4167869, 1074)
Prototype matrix: (25, 1074)
Cell types: ['neuroblast (sensu Vertebrata)', 'ependymal cell', 'cholinergic neuron', 'endothelial cell', 'astrocyte', 'oligodendrocyte', 'microglial cell', 'smooth muscle cell', 'macrophage', 'dendritic cell', 'lymphocyte', 'monocyte', 'GABAergic neuron', 'Bergmann glial cell', 'pericyte', 'glutamatergic neuron', 'dopaminergic neuron', 'choroid plexus epithelial cell', 'tanycyte', 'oligodendrocyte precursor cell', 'olfactory ensheathing cell', 'histaminergic neuron', 'glycinergic neuron', 'vascular leptomeningeal cell', 'hypendymal cell']


In [15]:
# ============================================================
# CELL 11 — Validate Prototypes
# ============================================================

# Check for degenerate (all-zero) prototypes
zero_rows = (prototypes_raw.sum(axis=1) == 0)
if zero_rows.any():
    print("WARNING — zero prototypes:", zero_rows[zero_rows].index.tolist())
else:
    print("No degenerate prototypes. ✓")

print("\nTop 5 marker genes per cell type:")
for ct in prototypes_raw.index:
    top5 = prototypes_raw.loc[ct].nlargest(5).index.tolist()
    print(f"  {ct}: {top5}")

No degenerate prototypes. ✓

Top 5 marker genes per cell type:
  neuroblast (sensu Vertebrata): ['MARCKSL1', 'SOX11', 'MEIS2', 'NFIX', 'IGFBPL1']
  ependymal cell: ['SLC38A1', 'GPRC5B', 'GJA1', 'NFIX', 'PCP4L1']
  cholinergic neuron: ['NEFH', 'FGF1', 'PRPH', 'SV2C', 'GLRA1']
  endothelial cell: ['CLDN5', 'FN1', 'SLCO1C1', 'S1PR1', 'ATP10A']
  astrocyte: ['GJA1', 'GPR37L1', 'ACSBG1', 'NTSR2', 'S1PR1']
  oligodendrocyte: ['CLDN11', 'MOG', 'SOX10', 'GPRC5B', 'SEC14L5']
  microglial cell: ['CTSS', 'SALL1', 'MAFB', 'MAF', 'CORO1A']
  smooth muscle cell: ['ACTA2', 'MFGE8', 'MYLK', 'TNS1', 'TPM2']
  macrophage: ['IGF2', 'MAF', 'UCP2', 'IGFBP4', 'CTSS']
  dendritic cell: ['UCP2', 'CORO1A', 'PTPRC', 'LSP1', 'CTSS']
  lymphocyte: ['PTPRC', 'UCP2', 'CORO1A', 'ETS1', 'LCP1']
  monocyte: ['CORO1A', 'UCP2', 'PTPRC', 'LCP1', 'NFAM1']
  GABAergic neuron: ['SLC32A1', 'SLC38A1', 'GAD2', 'PCP4L1', 'NTRK3']
  Bergmann glial cell: ['GPR37L1', 'ACSBG1', 'SLC38A1', 'FAM107A', 'S1PR1']
  pericyte: ['MFGE8', '

In [16]:
# ============================================================
# CELL 12 — Row-Normalize Prototypes + Save All Outputs
# ============================================================

prototypes_norm_array = normalize(prototypes_raw.values, norm="l2", axis=1)
prototypes_norm = pd.DataFrame(
    prototypes_norm_array,
    index=prototypes_raw.index,
    columns=prototypes_raw.columns
)

prototypes_raw.to_csv(f"{OUT_DIR}/prototypes_raw.csv")
prototypes_norm.to_csv(f"{OUT_DIR}/prototypes_norm.csv")
np.save(f"{OUT_DIR}/prototypes_norm.npy", prototypes_norm_array)
adata_sc.write(f"{OUT_DIR}/adata_sc_processed.h5ad")

print("Saved: prototypes_raw.csv, prototypes_norm.csv, prototypes_norm.npy, adata_sc_processed.h5ad")
print("\n Part 1 complete! ✓")

Saved: prototypes_raw.csv, prototypes_norm.csv, prototypes_norm.npy, adata_sc_processed.h5ad

 Part 1 complete! ✓
